# Experiment 2: Reasoning Depth Scaling

How does accuracy degrade as the **length of the equality chain increases**?

## Setup
We generate equality chains of depth D:
- **Depth 2:** `"A=B, B=C. Does A=C?"` — one inference hop (A→B→C)
- **Depth 3:** `"A=B, B=C, C=D. Does A=D?"` — two hops
- **Depth D:** D premises, D−1 hops, final question asks about the endpoints

For negative examples, we **break the chain** at a random link with 'does not equal'.

## Why this matters
Genuine transitive reasoning should scale linearly. If LLMs rely on pattern-matching, accuracy will drop off sharply after depth 2–3 — revealing the shallow nature of their reasoning.

## Models tested
1. **Phi-3-mini zero-shot** — logit-based yes/no comparison
2. **Phi-3-mini LoRA fine-tuned** — if the `phi3_lora_out` checkpoint exists

## Key metrics
- Accuracy per depth
- Accuracy on positive examples (chain intact) vs. negative (chain broken)
- Depth at which model drops below 60% / 55% / 50% (random chance)

In [ ]:
# ── Setup: clone repo and install dependencies ──────────────────────────────
!git clone https://github.com/Atharvy700/SP-25.git 2>/dev/null || echo 'Already cloned'
%cd SP-25
!pip install transformers accelerate peft -q

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json, random, gc, os, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

random.seed(42)
np.random.seed(42)

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Generate entity name pool ────────────────────────────────────────────────
def gen_names(n=50000, lo=4, hi=7, seed=42):
    rng = random.Random(seed)
    names = set()
    while len(names) < n:
        k = rng.randint(lo, hi)
        names.add(''.join(rng.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=k)))
    return list(names)

NAMES = gen_names()
print(f'Name pool: {len(NAMES)} entities')
print(f'Examples: {NAMES[:8]}')

In [ ]:
# ── Chain data generation ─────────────────────────────────────────────────────

def gen_positive_chain(depth: int, names: list, rng: random.Random) -> dict:
    """
    Positive example: full equality chain of length `depth`.
    A0=A1=...=A_depth  →  question: Does A0 equal A_depth?  →  yes
    """
    chain = rng.sample(names, depth + 1)
    premises = [f'Suppose {chain[i]} equals {chain[i+1]}.' for i in range(depth)]
    question  = f'Does {chain[0]} equal {chain[-1]}?'
    text = ' '.join(premises) + ' ' + question
    return {'text': text, 'label': 'yes', 'depth': depth, 'type': 'positive', 'chain': chain}


def gen_negative_chain(depth: int, names: list, rng: random.Random) -> dict:
    """
    Negative example: chain broken at one random link.
    One premise uses 'does not equal' → transitivity fails → answer: no
    """
    chain = rng.sample(names, depth + 1)
    break_at = rng.randint(0, depth - 1)   # which link to break
    premises = []
    for i in range(depth):
        if i == break_at:
            premises.append(f'Suppose {chain[i]} does not equal {chain[i+1]}.')
        else:
            premises.append(f'Suppose {chain[i]} equals {chain[i+1]}.')
    question = f'Does {chain[0]} equal {chain[-1]}?'
    text = ' '.join(premises) + ' ' + question
    return {'text': text, 'label': 'no', 'depth': depth, 'type': 'negative',
            'chain': chain, 'break_at': break_at}


def gen_dataset(depths: list, n_per_depth: int = 200, seed: int = 42) -> dict:
    """
    Generate balanced (50/50) datasets for each depth.
    Returns dict: {depth: [examples]}
    """
    rng = random.Random(seed)
    datasets = {}
    for d in depths:
        pos = [gen_positive_chain(d, NAMES, rng) for _ in range(n_per_depth // 2)]
        neg = [gen_negative_chain(d, NAMES, rng) for _ in range(n_per_depth // 2)]
        examples = pos + neg
        rng.shuffle(examples)
        datasets[d] = examples
        print(f'  Depth {d}: {len(examples)} examples ({len(pos)} pos, {len(neg)} neg)')
    return datasets

# ── CONFIG ───────────────────────────────────────────────────────────────────
DEPTHS      = [2, 3, 4, 5, 6, 7, 8, 10]  # chain depths to test
N_PER_DEPTH = 200                          # 100 pos + 100 neg per depth

print('Generating datasets...')
datasets = gen_dataset(DEPTHS, N_PER_DEPTH)

# Preview an example at each depth
print()
for d in [2, 4, 8]:
    ex = datasets[d][0]
    print(f'--- Depth {d} ({ex["label"]}) ---')
    print(ex['text'])
    print()

In [ ]:
# ── Load Phi-3-mini zero-shot model ──────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'
MAX_LENGTH  = 512   # longer for deep chains

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model (FP16)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
base_model.eval()

YES_ID = tokenizer('yes', add_special_tokens=False).input_ids[0]
NO_ID  = tokenizer('no',  add_special_tokens=False).input_ids[0]
print(f'Ready. yes_id={YES_ID}, no_id={NO_ID}')

In [ ]:
# ── Inference helpers ────────────────────────────────────────────────────────

def build_prompt(text: str) -> str:
    return (
        'You are a logical reasoning assistant.\n'
        'Answer with exactly one word.\n\n'
        f'{text}\n\n'
        'Answer only yes or no:'
    )

@torch.inference_mode()
def predict_batch(model, prompts: list, batch_size: int = 4) -> list:
    """Logit-based yes/no prediction in batches."""
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(model.device)
        logits = model(**inputs).logits[:, -1, :]   # [B, vocab]
        for logit in logits:
            results.append('yes' if logit[YES_ID] > logit[NO_ID] else 'no')
    return results


def evaluate_model_on_depths(model, datasets: dict, model_label: str) -> dict:
    """Run the model over all depths and return per-depth metrics."""
    results = {}
    for depth, examples in datasets.items():
        prompts = [build_prompt(ex['text']) for ex in examples]
        gold    = [ex['label'] for ex in examples]
        pos_idx = [i for i, ex in enumerate(examples) if ex['type'] == 'positive']
        neg_idx = [i for i, ex in enumerate(examples) if ex['type'] == 'negative']

        preds = predict_batch(model, prompts)

        overall_acc = np.mean([p == g for p, g in zip(preds, gold)])
        pos_acc     = np.mean([preds[i] == gold[i] for i in pos_idx]) if pos_idx else None
        neg_acc     = np.mean([preds[i] == gold[i] for i in neg_idx]) if neg_idx else None

        # Bias: does the model always say 'yes' regardless?
        yes_rate    = np.mean([p == 'yes' for p in preds])

        results[depth] = {
            'overall_acc': float(overall_acc),
            'pos_acc':     float(pos_acc)     if pos_acc     is not None else None,
            'neg_acc':     float(neg_acc)     if neg_acc     is not None else None,
            'yes_rate':    float(yes_rate),
            'n':           len(examples),
        }
        print(f'  [{model_label}] Depth {depth:2d}: '
              f'acc={overall_acc:.4f}  pos={pos_acc:.4f}  neg={neg_acc:.4f}  '
              f'yes_rate={yes_rate:.4f}')

    return results

print('Helpers defined.')

In [ ]:
# ── RUN: Phi-3 zero-shot ─────────────────────────────────────────────────────
print('Evaluating Phi-3-mini (zero-shot) across all depths...')
print()
zeroshot_results = evaluate_model_on_depths(base_model, datasets, 'zero-shot')

In [ ]:
# ── RUN: Phi-3 LoRA fine-tuned (if checkpoint exists) ───────────────────────
LORA_CHECKPOINT = './phi3_lora_out'

if os.path.exists(LORA_CHECKPOINT):
    print(f'Loading LoRA fine-tuned model from {LORA_CHECKPOINT}...')
    gc.collect()
    torch.cuda.empty_cache()

    finetuned_model = PeftModel.from_pretrained(
        base_model,
        LORA_CHECKPOINT,
        torch_dtype=torch.float16,
    )
    finetuned_model.eval()
    print('Fine-tuned model loaded.')
    print()
    print('Evaluating Phi-3-mini (fine-tuned) across all depths...')
    finetuned_results = evaluate_model_on_depths(finetuned_model, datasets, 'fine-tuned')
else:
    print(f'No fine-tuned checkpoint found at {LORA_CHECKPOINT}.')
    print('Only zero-shot results will be plotted.')
    print('To run fine-tuning first, execute Phi_Code_(1).ipynb.')
    finetuned_results = None

In [ ]:
# ── MAIN PLOT: Accuracy vs. Chain Depth ──────────────────────────────────────
depths      = sorted(zeroshot_results.keys())
zs_acc      = [zeroshot_results[d]['overall_acc'] for d in depths]
zs_pos_acc  = [zeroshot_results[d]['pos_acc']     for d in depths]
zs_neg_acc  = [zeroshot_results[d]['neg_acc']     for d in depths]
zs_yes_rate = [zeroshot_results[d]['yes_rate']    for d in depths]

if finetuned_results:
    ft_acc     = [finetuned_results[d]['overall_acc'] for d in depths]
    ft_pos_acc = [finetuned_results[d]['pos_acc']     for d in depths]
    ft_neg_acc = [finetuned_results[d]['neg_acc']     for d in depths]

# ── Figure 1: Overall accuracy ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(depths, zs_acc, 'o-', color='#2196F3', linewidth=2.5, markersize=9,
        label='Phi-3-mini (zero-shot)', zorder=5)

if finetuned_results:
    ax.plot(depths, ft_acc, 's--', color='#FF5722', linewidth=2.5, markersize=9,
            label='Phi-3-mini (LoRA fine-tuned)', zorder=5)

ax.axhline(0.5,  color='red',   linestyle=':',  alpha=0.8, linewidth=1.5, label='Random chance (50%)')
ax.axhline(1.0,  color='green', linestyle='--', alpha=0.5, linewidth=1.5, label='Perfect (100%)')

# Annotate where model crosses 60%
for i, (d, acc) in enumerate(zip(depths, zs_acc)):
    ax.annotate(f'{acc:.2f}', (d, acc), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, color='#2196F3')

if finetuned_results:
    for i, (d, acc) in enumerate(zip(depths, ft_acc)):
        ax.annotate(f'{acc:.2f}', (d, acc), textcoords='offset points',
                    xytext=(0, -18), ha='center', fontsize=9, color='#FF5722')

ax.set_xlabel('Chain Depth (number of equality premises)', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title(
    'Reasoning Depth Scaling: Accuracy vs. Equality Chain Length\n'
    'Phi-3-mini-4k-instruct — 50/50 balanced test, random entity names',
    fontsize=13, fontweight='bold'
)
ax.set_xticks(depths)
ax.set_ylim(0.3, 1.1)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('experiment2_depth_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiment2_depth_overall.png')

In [ ]:
# ── FIGURE 2: Positive vs. Negative accuracy breakdown ───────────────────────
# This reveals model bias — does it collapse to always saying 'yes'?

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    'Positive (chain intact) vs. Negative (chain broken) Accuracy\n'
    'A model biased toward "yes" will have high pos_acc and low neg_acc',
    fontsize=12, fontweight='bold'
)

# Zero-shot breakdown
ax = axes[0]
ax.plot(depths, zs_pos_acc,  'o-', color='#4CAF50', linewidth=2.5, markersize=8,
        label='Positive examples (label=yes)')
ax.plot(depths, zs_neg_acc,  's-', color='#F44336', linewidth=2.5, markersize=8,
        label='Negative examples (label=no)')
ax.plot(depths, zs_acc,      'd--', color='#2196F3', linewidth=2, markersize=6,
        alpha=0.7, label='Overall accuracy')
ax.plot(depths, zs_yes_rate, '^:', color='#FF9800', linewidth=2, markersize=6,
        alpha=0.8, label='"Yes" output rate')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.8, linewidth=1.5)
ax.set_xlabel('Chain Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Phi-3-mini (zero-shot)', fontsize=12, fontweight='bold')
ax.set_xticks(depths)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Fine-tuned breakdown
ax = axes[1]
if finetuned_results:
    ft_yes_rate = [finetuned_results[d]['yes_rate'] for d in depths]
    ax.plot(depths, ft_pos_acc,  'o-', color='#4CAF50', linewidth=2.5, markersize=8,
            label='Positive examples (label=yes)')
    ax.plot(depths, ft_neg_acc,  's-', color='#F44336', linewidth=2.5, markersize=8,
            label='Negative examples (label=no)')
    ax.plot(depths, ft_acc,      'd--', color='#FF5722', linewidth=2, markersize=6,
            alpha=0.7, label='Overall accuracy')
    ax.plot(depths, ft_yes_rate, '^:', color='#FF9800', linewidth=2, markersize=6,
            alpha=0.8, label='"Yes" output rate')
    ax.set_title('Phi-3-mini (LoRA fine-tuned)', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Fine-tuned model not available.\nRun Phi_Code_(1).ipynb first.',
            ha='center', va='center', transform=ax.transAxes, fontsize=12,
            color='gray')
    ax.set_title('Phi-3-mini (LoRA fine-tuned) — N/A', fontsize=12)

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.8, linewidth=1.5)
ax.set_xlabel('Chain Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_xticks(depths)
ax.set_ylim(0, 1.1)
if finetuned_results:
    ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiment2_posneg_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiment2_posneg_breakdown.png')

In [ ]:
# ── FIGURE 3: Accuracy degradation curve (normalized to depth-2 baseline) ────
# How much does each additional hop hurt performance?

fig, ax = plt.subplots(figsize=(10, 5))

base_zs = zs_acc[0]   # depth-2 accuracy = baseline
zs_normalized = [acc / base_zs for acc in zs_acc]

ax.plot(depths, zs_normalized, 'o-', color='#2196F3', linewidth=2.5, markersize=9,
        label='Zero-shot (normalized to depth-2)')

if finetuned_results:
    base_ft = ft_acc[0]
    ft_normalized = [acc / base_ft for acc in ft_acc]
    ax.plot(depths, ft_normalized, 's--', color='#FF5722', linewidth=2.5, markersize=9,
            label='Fine-tuned (normalized to depth-2)')

ax.axhline(1.0, color='green', linestyle='--', alpha=0.5, linewidth=1.5, label='No degradation')
ax.axhline(0.5 / base_zs, color='red', linestyle=':', alpha=0.7, linewidth=1.5,
           label=f'Random chance ({0.5/base_zs:.2f}x baseline)')

ax.set_xlabel('Chain Depth', fontsize=12)
ax.set_ylabel('Relative Accuracy (vs. Depth-2)', fontsize=12)
ax.set_title('Accuracy Degradation with Chain Depth\n(1.0 = same as depth-2 performance)', fontsize=12, fontweight='bold')
ax.set_xticks(depths)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('experiment2_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiment2_degradation.png')

In [ ]:
# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print('=' * 75)
print('DEPTH SCALING RESULTS — Phi-3-mini-4k-instruct')
print('=' * 75)

header = f'{"Depth":>6}  {"ZS Overall":>11} {"ZS Pos":>8} {"ZS Neg":>8} {"ZS Yes%":>8}'
if finetuned_results:
    header += f'  {"FT Overall":>11} {"FT Pos":>8} {"FT Neg":>8}'
print(header)
print('-' * (75 if not finetuned_results else 110))

for d in depths:
    z = zeroshot_results[d]
    row = (f'{d:>6}  {z["overall_acc"]:>11.4f} {z["pos_acc"]:>8.4f} '
           f'{z["neg_acc"]:>8.4f} {z["yes_rate"]:>8.4f}')
    if finetuned_results:
        f = finetuned_results[d]
        row += f'  {f["overall_acc"]:>11.4f} {f["pos_acc"]:>8.4f} {f["neg_acc"]:>8.4f}'
    print(row)

print('=' * 75)
print()

# Find the depth at which zero-shot drops below key thresholds
print('Zero-shot performance thresholds:')
for threshold in [0.80, 0.70, 0.60, 0.55]:
    crossed = next((d for d, acc in zip(depths, zs_acc) if acc < threshold), None)
    if crossed:
        print(f'  Drops below {threshold:.0%} at depth {crossed}')
    else:
        print(f'  Never drops below {threshold:.0%} in tested range')

if finetuned_results:
    print()
    print('Fine-tuned performance thresholds:')
    for threshold in [0.80, 0.70, 0.60, 0.55]:
        crossed = next((d for d, acc in zip(depths, ft_acc) if acc < threshold), None)
        if crossed:
            print(f'  Drops below {threshold:.0%} at depth {crossed}')
        else:
            print(f'  Never drops below {threshold:.0%} in tested range')

print()
print('Interpretation:')
print('  pos_acc = accuracy on chains that ARE intact (should say yes)')
print('  neg_acc = accuracy on chains that ARE broken (should say no)')
print('  yes_rate >> 0.5 → model is biased toward "yes" (pattern-matching)')
print('  If pos_acc >> neg_acc at large depths → model ignores the break')
print('    and defaults to "yes" when it sees a long chain')

In [ ]:
# ── Show example prompts at each depth ───────────────────────────────────────
print('EXAMPLE PROMPTS BY DEPTH')
print('=' * 70)
for d in [2, 4, 6, 8]:
    ex = datasets[d][0]
    print(f'\n--- Depth {d} | label={ex["label"]} ---')
    print(build_prompt(ex['text']))
    print()

In [ ]:
# ── Save results to JSON ─────────────────────────────────────────────────────
results_out = {
    'model': MODEL_NAME,
    'depths': depths,
    'n_per_depth': N_PER_DEPTH,
    'zeroshot': {
        str(d): zeroshot_results[d] for d in depths
    },
}
if finetuned_results:
    results_out['finetuned'] = {
        str(d): finetuned_results[d] for d in depths
    }

with open('experiment2_results.json', 'w') as f:
    json.dump(results_out, f, indent=2)

print('Results saved to experiment2_results.json')
print(json.dumps(results_out, indent=2))